# Master ML Model Evaluator

Compare averaged results from the three learned approaches:

- **RL**: re-evaluates every listed Stable Baselines `.zip` model on the selected RL v4 split.
- **Multiclass**: loads the saved prediction CSV from every listed output directory.
- **LLM**: loads the saved prediction CSV from every listed output directory.

The notebook also reports no optimization, the best fixed reward chain selected on the training split, a uniform random-chain expectation, the per-circuit reward oracle, and one metric-specific oracle in each metric's own table and chart.

In [ ]:
from pathlib import Path
import json
import os
import sys
import warnings

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from stable_baselines3 import A2C, DQN, PPO

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "dbs" / "combined_master.csv").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate the project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RL_DIR = PROJECT_ROOT / "machine_learning" / "RL" / "two_opt_chain_rl_v4"
if str(RL_DIR) not in sys.path:
    sys.path.insert(0, str(RL_DIR))

from data_processing import feature_batch_names, split_summary_frame
from evaluation import evaluate_model_on_frame
from rewards import (
    SUPPORTED_REWARD_TYPES,
    compute_final_correction,
    incremental_step_reward,
    single_step_reward,
)
from train import build_env, reward_splits

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

ALGORITHM_CLASSES = {"ppo": PPO, "a2c": A2C, "dqn": DQN}
FEATURE_BATCH_ALIASES = {
    "all_features": "plus_all_features",
    "plus_all_no_history": "plus_all_features",
}

## Configuration

Edit only this cell for a comparison.

The RL and LLM runs must match all five reward fields. The multiclass model was intentionally designed without a final reward correction, so it is checked against the three weights it supports and must declare `final_reward_correction: false`.

`Random Chain` is the exact expected result of choosing uniformly from all 21 valid actions for each circuit, including no optimization. It is deterministic, not one sampled random trial.

In [ ]:
# Shared evaluation setup

# 2q = 1.0, depth = 0, runtime = 0, final reward = 1.0
EVAL_SPLIT = "test"  # "validation" or "test"
REWARD_TYPE = "mixed_weighted"
TWO_QUBIT_WEIGHT = 1.0
DEPTH_WEIGHT = 0.0
RUNTIME_WEIGHT = 0.0
FINAL_REWARD_WEIGHT = 1.0

# RL settings and model files. All files must be represented in the RL training log.
RL_ALGORITHM = "ppo"
RL_FEATURE_BATCH = "plus_all_features"
RL_TRAINING_LOG_PATH = RL_DIR / "grid_search_runs_rewards.csv"
RL_MODEL_PATHS = [
    RL_DIR / "models" / "optimizer_chain_ppo_mixed_weighted_seed42_fbplus_all_features_rw0p0_tqw1p0_dw0p0_frw1p0.zip",
    RL_DIR / "models" / "optimizer_chain_ppo_mixed_weighted_seed43_fbplus_all_features_rw0p0_tqw1p0_dw0p0_frw1p0.zip",
    RL_DIR / "models" / "optimizer_chain_ppo_mixed_weighted_seed44_fbplus_all_features_rw0p0_tqw1p0_dw0p0_frw1p0.zip",
]

# Saved multiclass output directories. Each needs run_config.json and the selected split's predictions CSV.
# Add more seeds after generating them with the fixed RL v4 split seed (42).
MULTICLASS_RUN_DIRS = [
    PROJECT_ROOT / 'machine_learning' / 'supervised_learning' / 'multiclass' / 'artifacts' / 'best_chain_random_forest_2q1_depth0_rt0_seed42',
    PROJECT_ROOT / 'machine_learning' / 'supervised_learning' / 'multiclass' / 'artifacts' / 'best_chain_random_forest_2q1_depth0_rt0_seed43',
    PROJECT_ROOT / 'machine_learning' / 'supervised_learning' / 'multiclass' / 'artifacts' / 'best_chain_random_forest_2q1_depth0_rt0_seed44'
]

# Saved LLM output directories. Each needs manifest.json and predictions.csv.
LLM_RUN_DIRS = [
    PROJECT_ROOT / "machine_learning" / "LLM" / "llm_v3" / "artifacts" / "full_system_test_2q_only",
]



# 2q = 1.0, depth = 0.1, runtime = 0.1, final reward = 1.0
#EVAL_SPLIT = "test"  # "validation" or "test"
#REWARD_TYPE = "mixed_weighted"
#TWO_QUBIT_WEIGHT = 1.0
#DEPTH_WEIGHT = 0.1
#RUNTIME_WEIGHT = 0.1
#FINAL_REWARD_WEIGHT = 1.0

# RL settings and model files. All files must be represented in the RL training log.
RL_ALGORITHM = "ppo"
RL_FEATURE_BATCH = "plus_all_features"
RL_TRAINING_LOG_PATH = RL_DIR / "grid_search_runs_rewards.csv"
#RL_MODEL_PATHS = [
#    RL_DIR / "models" / "optimizer_chain_ppo_mixed_weighted_seed42_fbplus_all_features_rw0p1_tqw1p0_dw0p1_frw1p0.zip",
#    RL_DIR / "models" / "optimizer_chain_ppo_mixed_weighted_seed43_fbplus_all_features_rw0p1_tqw1p0_dw0p1_frw1p0.zip",
#    RL_DIR / "models" / "optimizer_chain_ppo_mixed_weighted_seed44_fbplus_all_features_rw0p1_tqw1p0_dw0p1_frw1p0.zip",
#]

# Saved multiclass output directories. Each needs run_config.json and the selected split's predictions CSV.
# Add more seeds after generating them with the fixed RL v4 split seed (42).
#MULTICLASS_RUN_DIRS = [
#    PROJECT_ROOT / 'machine_learning' / 'supervised_learning' / 'multiclass' / 'artifacts' / 'best_chain_random_forest_2q1_depth0.1_rt0.1_seed42',
#    PROJECT_ROOT / 'machine_learning' / 'supervised_learning' / 'multiclass' / 'artifacts' / 'best_chain_random_forest_2q1_depth0.1_rt0.1_seed43',
#    PROJECT_ROOT / 'machine_learning' / 'supervised_learning' / 'multiclass' / 'artifacts' / 'best_chain_random_forest_2q1_depth0.1_rt0.1_seed44'
#]

# Saved LLM output directories. Each needs manifest.json and predictions.csv.
#LLM_RUN_DIRS = [
#    PROJECT_ROOT / "machine_learning" / "LLM" / "llm_v3" / "artifacts" / "full_system_test",
#]

SAVE_FIGURES = True
FIGURE_DIR = PROJECT_ROOT / "machine_learning" / "master_ml_model_evaluator_figures"

RL_FEATURE_BATCH = FEATURE_BATCH_ALIASES.get(RL_FEATURE_BATCH, RL_FEATURE_BATCH)
RL_ALGORITHM = RL_ALGORITHM.lower()

if EVAL_SPLIT not in {"validation", "test"}:
    raise ValueError('EVAL_SPLIT must be "validation" or "test".')
if REWARD_TYPE not in SUPPORTED_REWARD_TYPES:
    raise ValueError(f"Unsupported REWARD_TYPE={REWARD_TYPE!r}.")
if RL_ALGORITHM not in ALGORITHM_CLASSES:
    raise ValueError(f"Unsupported RL_ALGORITHM={RL_ALGORITHM!r}.")
if RL_FEATURE_BATCH not in feature_batch_names():
    raise ValueError(f"Unsupported RL_FEATURE_BATCH={RL_FEATURE_BATCH!r}.")
if not RL_MODEL_PATHS or not MULTICLASS_RUN_DIRS or not LLM_RUN_DIRS:
    raise ValueError("Provide at least one RL model, multiclass run, and LLM run.")

EXPECTED_REWARD = {
    "reward_type": REWARD_TYPE,
    "runtime_weight": float(RUNTIME_WEIGHT),
    "two_qubit_weight": float(TWO_QUBIT_WEIGHT),
    "depth_weight": float(DEPTH_WEIGHT),
    "final_reward_weight": float(FINAL_REWARD_WEIGHT),
}

display(pd.DataFrame([{
    "split": EVAL_SPLIT,
    **EXPECTED_REWARD,
    "rl_feature_batch": RL_FEATURE_BATCH,
    "rl_models": len(RL_MODEL_PATHS),
    "multiclass_runs": len(MULTICLASS_RUN_DIRS),
    "llm_runs": len(LLM_RUN_DIRS),
}]))

## Load and validate runs

This section raises an error before comparison if a path is missing, a split is wrong, a run is incomplete, the evaluated circuits differ from the RL v4 split, or the saved reward settings do not match the configuration above.

In [ ]:
METRIC_COLUMNS = {
    "two_qubit_reduction_pct": "2Q gate reduction (%)",
    "depth_reduction_pct": "Depth reduction (%)",
    "runtime_seconds": "Runtime (seconds)",
}


def resolved_paths(paths, kind):
    result = [Path(path).expanduser().resolve() for path in paths]
    missing = [str(path) for path in result if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing {kind}: {missing}")
    return result


def values_match(actual, expected):
    if isinstance(expected, (float, int)) and not isinstance(expected, bool):
        try:
            return bool(np.isclose(float(actual), float(expected), atol=1e-9, rtol=0.0))
        except (TypeError, ValueError):
            return False
    return actual == expected


def require_matching_values(actual, expected, source):
    mismatches = {
        key: {"expected": expected_value, "actual": actual.get(key)}
        for key, expected_value in expected.items()
        if not values_match(actual.get(key), expected_value)
    }
    if mismatches:
        raise ValueError(f"{source} has incompatible settings: {mismatches}")


def standardize_metrics(
    frame,
    circuit_column,
    twoq_column,
    depth_column,
    runtime_column,
    chain_column=None,
):
    result = pd.DataFrame({
        "circuit_id": frame[circuit_column].astype(str),
        "two_qubit_reduction_pct": pd.to_numeric(frame[twoq_column], errors="raise"),
        "depth_reduction_pct": pd.to_numeric(frame[depth_column], errors="raise"),
        "runtime_seconds": pd.to_numeric(frame[runtime_column], errors="raise"),
    })
    if chain_column is not None:
        result["selected_chain"] = frame[chain_column].fillna("__original__").astype(str)
    if result["circuit_id"].duplicated().any():
        duplicates = result.loc[result["circuit_id"].duplicated(), "circuit_id"].tolist()
        raise ValueError(f"Duplicate prediction rows for circuits: {duplicates[:10]}")
    return result.sort_values("circuit_id").reset_index(drop=True)


def require_expected_circuits(frame, expected_circuits, source):
    actual_circuits = set(frame["circuit_id"])
    if actual_circuits != expected_circuits:
        missing = sorted(expected_circuits - actual_circuits)
        extra = sorted(actual_circuits - expected_circuits)
        raise ValueError(
            f"{source} does not match the RL v4 {EVAL_SPLIT} split. "
            f"Missing circuits: {missing[:10]}; extra circuits: {extra[:10]}"
        )


def summarize_metrics(frame):
    return {
        metric: float(pd.to_numeric(frame[metric], errors="raise").mean())
        for metric in METRIC_COLUMNS
    }


def average_run_summaries(run_frames, method):
    summaries = pd.DataFrame([summarize_metrics(frame) for frame in run_frames])
    return {"method": method, **summaries.mean().to_dict()}


def load_rl_training_rows(model_paths):
    if not RL_TRAINING_LOG_PATH.exists():
        raise FileNotFoundError(f"RL training log not found: {RL_TRAINING_LOG_PATH}")

    log = pd.read_csv(RL_TRAINING_LOG_PATH)
    if "algorithm_name" in log.columns and "algorithm_type" not in log.columns:
        log = log.rename(columns={"algorithm_name": "algorithm_type"})
    if "status" in log.columns:
        log = log[log["status"].astype(str).str.lower() == "success"].copy()

    rows = []
    for model_path in model_paths:
        exact = log["model_path"].astype(str).map(
            lambda value: Path(value).expanduser().resolve() == model_path
        )
        matches = log[exact].copy()
        if matches.empty:
            matches = log[
                log["model_path"].astype(str).map(lambda value: Path(value).name == model_path.name)
            ].copy()
        if matches.empty:
            raise ValueError(f"No successful RL training-log row found for {model_path}.")

        row = matches.iloc[-1].copy()
        actual = {
            "reward_type": str(row["reward_type"]),
            "runtime_weight": float(row["runtime_weight"]),
            "two_qubit_weight": float(row["two_qubit_weight"]),
            "depth_weight": float(row["depth_weight"]),
            "final_reward_weight": float(row["final_reward_weight"]),
            "algorithm_type": str(row["algorithm_type"]).lower(),
            "feature_batch": FEATURE_BATCH_ALIASES.get(str(row["feature_batch"]), str(row["feature_batch"])),
        }
        require_matching_values(
            actual,
            {
                **EXPECTED_REWARD,
                "algorithm_type": RL_ALGORITHM,
                "feature_batch": RL_FEATURE_BATCH,
            },
            f"RL model {model_path.name}",
        )
        rows.append({"model_path": model_path, **actual})
    return rows


def load_multiclass_runs(run_dirs, expected_circuits):
    loaded = []
    prediction_name = f"{EVAL_SPLIT}_predictions.csv"
    expected = {
        "runtime_weight": RUNTIME_WEIGHT,
        "two_qubit_weight": TWO_QUBIT_WEIGHT,
        "depth_weight": DEPTH_WEIGHT,
        "final_reward_correction": False,
    }

    for run_dir in run_dirs:
        config_path = run_dir / "run_config.json"
        predictions_path = run_dir / prediction_name
        if not config_path.exists() or not predictions_path.exists():
            raise FileNotFoundError(
                f"Multiclass run {run_dir} needs run_config.json and {prediction_name}."
            )
        config = json.loads(config_path.read_text())
        require_matching_values(config, expected, f"Multiclass run {run_dir.name}")

        predictions = pd.read_csv(predictions_path)
        if "split" in predictions and set(predictions["split"].astype(str)) != {EVAL_SPLIT}:
            raise ValueError(f"Multiclass run {run_dir.name} contains the wrong split.")
        standardized = standardize_metrics(
            predictions,
            "original_circuit",
            "total_two_qubit_gate_reduction_pct",
            "total_depth_reduction_pct",
            "total_runtime_seconds",
            "predicted_chain",
        )
        require_expected_circuits(standardized, expected_circuits, f"Multiclass run {run_dir.name}")
        loaded.append(standardized)
    return loaded


def load_llm_runs(run_dirs, expected_circuits):
    loaded = []
    for run_dir in run_dirs:
        manifest_path = run_dir / "manifest.json"
        predictions_path = run_dir / "predictions.csv"
        if not manifest_path.exists() or not predictions_path.exists():
            raise FileNotFoundError(f"LLM run {run_dir} needs manifest.json and predictions.csv.")

        manifest = json.loads(manifest_path.read_text())
        if str(manifest.get("split")) != EVAL_SPLIT:
            raise ValueError(
                f"LLM run {run_dir.name} uses split={manifest.get('split')!r}, "
                f"expected {EVAL_SPLIT!r}."
            )
        require_matching_values(
            dict(manifest.get("objective", {})),
            EXPECTED_REWARD,
            f"LLM run {run_dir.name}",
        )

        predictions = pd.read_csv(predictions_path)
        standardized = standardize_metrics(
            predictions,
            "circuit_id",
            "two_qubit_gate_reduction_pct",
            "depth_reduction_pct",
            "runtime_seconds",
            "selected_chain",
        )
        require_expected_circuits(standardized, expected_circuits, f"LLM run {run_dir.name}")
        loaded.append(standardized)
    return loaded

## Rebuild the RL v4 split and reward values

The split and baseline candidates come directly from RL v4. The best fixed reward chain is chosen using training circuits only. Oracles are then computed per circuit on the selected evaluation split.

In [ ]:
bundle = reward_splits(
    runtime_weight=RUNTIME_WEIGHT,
    reward_type=REWARD_TYPE,
    two_qubit_weight=TWO_QUBIT_WEIGHT,
    depth_weight=DEPTH_WEIGHT,
    save_splits=False,
    feature_batch=RL_FEATURE_BATCH,
)

optimizer_names = bundle["optimizer_names"]
category_names = bundle["category_names"]
metric_scale = bundle["metric_scale"]
observation_feature_scale = bundle["observation_feature_scale"]
static_feature_columns = bundle["static_feature_columns"]
dynamic_feature_columns = bundle["dynamic_feature_columns"]
include_category = bundle["feature_batch_config"]["include_category"]


def add_episode_rewards(frame):
    result = frame.copy()
    row_lookup = {
        (str(row.original_circuit), str(row.chain_name)): row._asdict()
        for row in result.itertuples(index=False)
    }
    rewards = []

    for row in result.itertuples(index=False):
        values = row._asdict()
        chain_name = str(row.chain_name)
        if "__" not in chain_name:
            reward = single_step_reward(
                values,
                reward_type=REWARD_TYPE,
                runtime_weight=RUNTIME_WEIGHT,
                two_qubit_weight=TWO_QUBIT_WEIGHT,
                depth_weight=DEPTH_WEIGHT,
            )
        else:
            first_optimizer = chain_name.split("__", 1)[0]
            first_values = row_lookup[(str(row.original_circuit), first_optimizer)]
            reward = single_step_reward(
                first_values,
                reward_type=REWARD_TYPE,
                runtime_weight=RUNTIME_WEIGHT,
                two_qubit_weight=TWO_QUBIT_WEIGHT,
                depth_weight=DEPTH_WEIGHT,
            )
            reward += incremental_step_reward(
                values,
                first_values,
                reward_type=REWARD_TYPE,
                runtime_weight=RUNTIME_WEIGHT,
                two_qubit_weight=TWO_QUBIT_WEIGHT,
                depth_weight=DEPTH_WEIGHT,
            )

        reward += compute_final_correction(
            values,
            reward_type=REWARD_TYPE,
            final_reward_weight=FINAL_REWARD_WEIGHT,
            two_qubit_weight=TWO_QUBIT_WEIGHT,
            depth_weight=DEPTH_WEIGHT,
        )
        rewards.append(float(reward))

    result["episode_reward_if_selected"] = rewards
    return result


def original_candidate(source_row):
    row = source_row.copy()
    row["chain_name"] = "__original__"
    row["total_two_qubit_gate_reduction_pct"] = 0.0
    row["total_depth_reduction_pct"] = 0.0
    row["total_runtime_seconds"] = 0.0
    row["total_runtime_score"] = 0.0
    row["episode_reward_if_selected"] = 0.0
    return row


def add_original_candidates(frame):
    originals = pd.DataFrame([
        original_candidate(group.iloc[0])
        for _, group in frame.groupby("original_circuit", sort=True)
    ])
    return pd.concat([frame, originals], ignore_index=True)


train_df = add_episode_rewards(bundle["train_df"])
validation_df = add_episode_rewards(bundle["validation_df"])
test_df = add_episode_rewards(bundle["test_df"])
eval_df = {"validation": validation_df, "test": test_df}[EVAL_SPLIT].copy()

train_candidates = add_original_candidates(train_df)
eval_candidates = add_original_candidates(eval_df)
expected_circuits = set(eval_df["original_circuit"].astype(str))

display(split_summary_frame({
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}))

## Evaluate and average the three learned methods

Only one averaged row per method is produced. Individual model/run results are used internally but are not displayed.

In [ ]:
rl_model_paths = resolved_paths(RL_MODEL_PATHS, "RL model files")
multiclass_run_dirs = resolved_paths(MULTICLASS_RUN_DIRS, "multiclass run directories")
llm_run_dirs = resolved_paths(LLM_RUN_DIRS, "LLM run directories")
rl_training_rows = load_rl_training_rows(rl_model_paths)

rl_run_frames = []
for metadata in rl_training_rows:
    model = ALGORITHM_CLASSES[RL_ALGORITHM].load(str(metadata["model_path"]))
    selected = evaluate_model_on_frame(
        model,
        eval_df,
        build_env,
        optimizer_names,
        category_names,
        metric_scale,
        observation_feature_scale,
        static_feature_columns,
        dynamic_feature_columns,
        include_category,
        method_name="RL",
        reward_type=REWARD_TYPE,
        runtime_weight=RUNTIME_WEIGHT,
        two_qubit_weight=TWO_QUBIT_WEIGHT,
        depth_weight=DEPTH_WEIGHT,
        final_reward_weight=FINAL_REWARD_WEIGHT,
        verbose=False,
    )
    standardized = standardize_metrics(
        selected,
        "original_circuit",
        "total_two_qubit_gate_reduction_pct",
        "total_depth_reduction_pct",
        "total_runtime_seconds",
        "selected_chain",
    )
    require_expected_circuits(standardized, expected_circuits, metadata["model_path"].name)
    rl_run_frames.append(standardized)

multiclass_run_frames = load_multiclass_runs(multiclass_run_dirs, expected_circuits)
llm_run_frames = load_llm_runs(llm_run_dirs, expected_circuits)

learned_comparison = pd.DataFrame([
    average_run_summaries(rl_run_frames, "RL"),
    average_run_summaries(multiclass_run_frames, "RF Classifier"),
    average_run_summaries(llm_run_frames, "LLM"),
])

validation_summary = pd.DataFrame([
    {
        "method": "RL",
        "runs": len(rl_run_frames),
        "source": "Re-evaluated .zip models",
        "reward_check": "Full objective matched",
    },
    {
        "method": "RF Classifier",
        "runs": len(multiclass_run_frames),
        "source": "Saved predictions",
        "reward_check": "3 weights matched; final correction disabled",
    },
    {
        "method": "LLM",
        "runs": len(llm_run_frames),
        "source": "Saved predictions",
        "reward_check": "Full objective matched",
    },
])

display(validation_summary)
display(
    learned_comparison
    .rename(columns=METRIC_COLUMNS)
    .set_index("method")
    .style.format("{:.4f}")
)

## Baselines and oracles

- **No Optimization** keeps all reductions and runtime at zero.
- **Best Fixed Reward Chain** is the one optimization chain with the highest mean configured reward on training circuits.
- **Random Chain** is the uniform expected value across all 21 actions on every evaluation circuit.
- **Reward Oracle** chooses the highest configured reward separately for each evaluation circuit.
- The 2Q, depth, and runtime oracles appear only in their own metric. Because no optimization is a valid action, the minimum-runtime oracle is identical to no optimization.

In [ ]:
def standardize_candidate_rows(frame):
    return standardize_metrics(
        frame,
        "original_circuit",
        "total_two_qubit_gate_reduction_pct",
        "total_depth_reduction_pct",
        "total_runtime_seconds",
    )


def oracle_rows(frame, column, maximize=True):
    grouped = frame.groupby("original_circuit")[column]
    indices = grouped.idxmax() if maximize else grouped.idxmin()
    return frame.loc[indices].copy()


best_fixed_reward_chain = str(
    train_df.groupby("chain_name")["episode_reward_if_selected"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)

no_optimization = eval_candidates[
    eval_candidates["chain_name"] == "__original__"
].copy()
best_fixed = eval_candidates[
    eval_candidates["chain_name"] == best_fixed_reward_chain
].copy()
reward_oracle = oracle_rows(eval_candidates, "episode_reward_if_selected", maximize=True)
twoq_oracle = oracle_rows(
    eval_candidates, "total_two_qubit_gate_reduction_pct", maximize=True
)
depth_oracle = oracle_rows(
    eval_candidates, "total_depth_reduction_pct", maximize=True
)
runtime_oracle = oracle_rows(
    eval_candidates, "total_runtime_seconds", maximize=False
)

random_expected_by_circuit = (
    eval_candidates.groupby("original_circuit", as_index=False)[
        [
            "total_two_qubit_gate_reduction_pct",
            "total_depth_reduction_pct",
            "total_runtime_seconds",
        ]
    ]
    .mean()
)
random_expected = standardize_metrics(
    random_expected_by_circuit,
    "original_circuit",
    "total_two_qubit_gate_reduction_pct",
    "total_depth_reduction_pct",
    "total_runtime_seconds",
)

shared_baselines = pd.DataFrame([
    {"method": "No Optimization", **summarize_metrics(standardize_candidate_rows(no_optimization))},
    {"method": "Best Fixed Reward Chain", **summarize_metrics(standardize_candidate_rows(best_fixed))},
    {"method": "Random Chain", **summarize_metrics(random_expected)},
    {"method": "Reward Oracle", **summarize_metrics(standardize_candidate_rows(reward_oracle))},
])

metric_oracles = {
    "two_qubit_reduction_pct": {
        "method": "2Q Oracle",
        **summarize_metrics(standardize_candidate_rows(twoq_oracle)),
    },
    "depth_reduction_pct": {
        "method": "Depth Oracle",
        **summarize_metrics(standardize_candidate_rows(depth_oracle)),
    },
    "runtime_seconds": {
        "method": "Runtime Oracle",
        **summarize_metrics(standardize_candidate_rows(runtime_oracle)),
    },
}

baseline_reference = pd.DataFrame([
    {
        "baseline": "Best Fixed Reward Chain",
        "selection source": "Training split",
        "selected chain": best_fixed_reward_chain,
    },
    {
        "baseline": "Random Chain",
        "selection source": f"Uniform expectation over {eval_candidates['chain_name'].nunique()} actions",
        "selected chain": "Varies",
    },
    {
        "baseline": "Reward Oracle",
        "selection source": f"Per-circuit {EVAL_SPLIT} upper bound",
        "selected chain": "Varies",
    },
])
display(baseline_reference)

## Metric tables and charts

Each learned method is averaged across its own list of runs. Shared baselines are repeated across the three comparisons, while each metric-specific oracle appears only in its matching comparison.

In [ ]:
METHOD_ORDER = [
    "RL",
    "RF Classifier",
    "LLM",
    "No Optimization",
    "Best Fixed Reward Chain",
    "Random Chain",
    "Reward Oracle",
]
METHOD_COLORS = {
    "RL": "#2A6F97",
    "RF Classifier": "#2A9D8F",
    "LLM": "#E76F51",
    "No Optimization": "#A8A8A8",
    "Best Fixed Reward Chain": "#6C757D",
    "Random Chain": "#B0BEC5",
    "Reward Oracle": "#264653",
    "2Q Oracle": "#E9C46A",
    "Depth Oracle": "#E9C46A",
    "Runtime Oracle": "#E9C46A",
}
METRIC_TITLES = {
    "two_qubit_reduction_pct": f"Average 2Q Gate Reduction on {EVAL_SPLIT.title()} Split",
    "depth_reduction_pct": f"Average Depth Reduction on {EVAL_SPLIT.title()} Split",
    "runtime_seconds": f"Average Runtime on {EVAL_SPLIT.title()} Split",
}

base_comparison = pd.concat(
    [learned_comparison, shared_baselines],
    ignore_index=True,
)
metric_tables = {}

if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

for metric, display_name in METRIC_COLUMNS.items():
    oracle = metric_oracles[metric]
    table = pd.concat(
        [
            base_comparison[["method", metric]],
            pd.DataFrame([{"method": oracle["method"], metric: oracle[metric]}]),
        ],
        ignore_index=True,
    )
    order = METHOD_ORDER + [oracle["method"]]
    table["method"] = pd.Categorical(table["method"], categories=order, ordered=True)
    table = table.sort_values("method").reset_index(drop=True)
    table["method"] = table["method"].astype(str)
    metric_tables[metric] = table

    print(METRIC_TITLES[metric])
    display(
        table.rename(columns={"method": "Method", metric: display_name})
        .set_index("Method")
        .style.format({display_name: "{:.4f}"})
    )

    fig, ax = plt.subplots(figsize=(10.5, 5.2))
    colors = [METHOD_COLORS[name] for name in table["method"]]
    bars = ax.bar(table["method"], table[metric], color=colors, edgecolor="white", linewidth=0.8)
    ax.axhline(0.0, color="#444444", linewidth=0.8)
    ax.set_title(METRIC_TITLES[metric], fontsize=14, pad=12)
    ax.set_ylabel(display_name)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=32)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")

    values = table[metric].to_numpy(dtype=float)
    value_range = float(np.nanmax(values) - np.nanmin(values))
    offset = 0.015 * (value_range if value_range > 0 else max(abs(values).max(), 1.0))
    for bar, value in zip(bars, values, strict=True):
        vertical_alignment = "bottom" if value >= 0 else "top"
        y = value + offset if value >= 0 else value - offset
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y,
            f"{value:.2f}",
            ha="center",
            va=vertical_alignment,
            fontsize=8.5,
        )

    sns.despine(ax=ax)
    fig.tight_layout()
    if SAVE_FIGURES:
        stem = {
            "two_qubit_reduction_pct": "two_qubit_reduction_comparison",
            "depth_reduction_pct": "depth_reduction_comparison",
            "runtime_seconds": "runtime_comparison",
        }[metric]
        fig.savefig(FIGURE_DIR / f"{stem}.png", dpi=200, bbox_inches="tight")
        fig.savefig(FIGURE_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()

if SAVE_FIGURES:
    print(f"Figures saved to: {FIGURE_DIR}")

## Performance by circuit size

Circuits are divided into balanced **Small**, **Medium**, and **Large** groups using their original total gate count before optimization. The range table shows the actual gate-count boundaries for the selected split.

Each value is first calculated within an individual model/run, then averaged across that method's runs.

In [ ]:
circuit_metadata = (
    eval_df[
        [
            "original_circuit",
            "original_category",
            "initial_total_gates",
            "initial_two_qubit_gates",
            "initial_num_qubits",
        ]
    ]
    .drop_duplicates("original_circuit")
    .rename(
        columns={
            "original_circuit": "circuit_id",
            "original_category": "circuit_type",
            "initial_total_gates": "original_total_gates",
            "initial_two_qubit_gates": "original_two_qubit_gates",
            "initial_num_qubits": "original_qubits",
        }
    )
    .sort_values("circuit_id")
    .reset_index(drop=True)
)
circuit_metadata["gate_size"] = pd.qcut(
    circuit_metadata["original_total_gates"].rank(method="first"),
    q=3,
    labels=["Small", "Medium", "Large"],
)

type_counts = circuit_metadata["circuit_type"].value_counts()
top_circuit_types = type_counts.head(10).index
circuit_metadata["circuit_type_plot"] = circuit_metadata["circuit_type"].where(
    circuit_metadata["circuit_type"].isin(top_circuit_types),
    "Other",
)

reward_lookup = eval_candidates[
    ["original_circuit", "chain_name", "episode_reward_if_selected"]
].rename(
    columns={
        "original_circuit": "circuit_id",
        "chain_name": "selected_chain",
        "episode_reward_if_selected": "selected_reward",
    }
)
oracle_reward_lookup = (
    eval_candidates.groupby("original_circuit", as_index=False)["episode_reward_if_selected"]
    .max()
    .rename(
        columns={
            "original_circuit": "circuit_id",
            "episode_reward_if_selected": "oracle_reward",
        }
    )
)


def enrich_selected_run(frame):
    enriched = frame.merge(
        reward_lookup,
        on=["circuit_id", "selected_chain"],
        how="left",
        validate="one_to_one",
    )
    enriched = enriched.merge(
        oracle_reward_lookup,
        on="circuit_id",
        how="left",
        validate="one_to_one",
    )
    enriched = enriched.merge(
        circuit_metadata,
        on="circuit_id",
        how="left",
        validate="one_to_one",
    )
    required = ["selected_reward", "oracle_reward", "gate_size", "circuit_type"]
    if enriched[required].isna().any().any():
        raise ValueError("Could not align selected chains with circuit metadata and oracle rewards.")
    enriched["regret_to_reward_oracle"] = (
        enriched["oracle_reward"] - enriched["selected_reward"]
    )
    enriched["selection_type"] = np.select(
        [
            enriched["selected_chain"].eq("__original__"),
            enriched["selected_chain"].str.contains("__", regex=False),
        ],
        ["Original", "Two-step chain"],
        default="Single optimizer",
    )
    return enriched


method_run_frames = {
    "RL": [enrich_selected_run(frame) for frame in rl_run_frames],
    "RF Classifier": [enrich_selected_run(frame) for frame in multiclass_run_frames],
    "LLM": [enrich_selected_run(frame) for frame in llm_run_frames],
}


def grouped_run_performance(group_column):
    rows = []
    for method, run_frames in method_run_frames.items():
        for run_index, frame in enumerate(run_frames):
            grouped = (
                frame.groupby(group_column, observed=True)
                .agg(
                    circuits=("circuit_id", "nunique"),
                    two_qubit_reduction_pct=("two_qubit_reduction_pct", "mean"),
                    depth_reduction_pct=("depth_reduction_pct", "mean"),
                    runtime_seconds=("runtime_seconds", "mean"),
                    regret_to_reward_oracle=("regret_to_reward_oracle", "mean"),
                )
                .reset_index()
            )
            grouped["method"] = method
            grouped["run_index"] = run_index
            rows.append(grouped)

    per_run = pd.concat(rows, ignore_index=True)
    return (
        per_run.groupby(["method", group_column], observed=True, as_index=False)
        .agg(
            circuits=("circuits", "mean"),
            two_qubit_reduction_pct=("two_qubit_reduction_pct", "mean"),
            depth_reduction_pct=("depth_reduction_pct", "mean"),
            runtime_seconds=("runtime_seconds", "mean"),
            regret_to_reward_oracle=("regret_to_reward_oracle", "mean"),
        )
    )


GROUP_METRICS = [
    ("two_qubit_reduction_pct", "2Q gate reduction (%)"),
    ("depth_reduction_pct", "Depth reduction (%)"),
    ("runtime_seconds", "Runtime (seconds)"),
    ("regret_to_reward_oracle", "Regret to reward oracle"),
]


def draw_group_performance(summary, group_column, group_order, title, filename):
    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
    for ax, (metric, label) in zip(axes.flat, GROUP_METRICS, strict=True):
        sns.barplot(
            data=summary,
            x=group_column,
            y=metric,
            hue="method",
            order=group_order,
            hue_order=["RL", "RF Classifier", "LLM"],
            palette=[METHOD_COLORS["RL"], METHOD_COLORS["RF Classifier"], METHOD_COLORS["LLM"]],
            errorbar=None,
            ax=ax,
        )
        ax.set_xlabel("")
        ax.set_ylabel(label)
        ax.tick_params(axis="x", rotation=35)
        if ax is not axes.flat[0] and ax.get_legend() is not None:
            ax.get_legend().remove()
        sns.despine(ax=ax)
    axes.flat[0].legend(title="", loc="best")
    fig.suptitle(title, fontsize=15, y=1.01)
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(FIGURE_DIR / f"{filename}.png", dpi=200, bbox_inches="tight")
        fig.savefig(FIGURE_DIR / f"{filename}.pdf", bbox_inches="tight")
    plt.show()


size_ranges = (
    circuit_metadata.groupby("gate_size", observed=True)
    .agg(
        circuits=("circuit_id", "nunique"),
        minimum_original_gates=("original_total_gates", "min"),
        maximum_original_gates=("original_total_gates", "max"),
        mean_original_gates=("original_total_gates", "mean"),
    )
    .reset_index()
)
size_performance = grouped_run_performance("gate_size")

display(size_ranges)
display(
    size_performance.rename(
        columns={
            "method": "Method",
            "gate_size": "Gate-size group",
            "circuits": "Circuits",
            "two_qubit_reduction_pct": "2Q reduction (%)",
            "depth_reduction_pct": "Depth reduction (%)",
            "runtime_seconds": "Runtime (s)",
            "regret_to_reward_oracle": "Reward regret",
        }
    )
    .set_index(["Gate-size group", "Method"])
    .style.format("{:.4f}")
)

draw_group_performance(
    size_performance,
    "gate_size",
    ["Small", "Medium", "Large"],
    f"Method Performance by Original Circuit Gate Count ({EVAL_SPLIT.title()})",
    "performance_by_gate_size",
)

## Performance by circuit type

The full table retains every original circuit category. For readability, the chart shows the ten most represented categories and combines the remaining rare categories into **Other**. Categories with one or two circuits should be interpreted cautiously.

In [ ]:
circuit_type_performance = grouped_run_performance("circuit_type")
circuit_type_plot_performance = grouped_run_performance("circuit_type_plot")
circuit_type_order = [
    name for name in type_counts.index if name in set(top_circuit_types)
] + ["Other"]

display(
    circuit_type_performance.rename(
        columns={
            "method": "Method",
            "circuit_type": "Circuit type",
            "circuits": "Circuits",
            "two_qubit_reduction_pct": "2Q reduction (%)",
            "depth_reduction_pct": "Depth reduction (%)",
            "runtime_seconds": "Runtime (s)",
            "regret_to_reward_oracle": "Reward regret",
        }
    )
    .sort_values(["Circuits", "Circuit type", "Method"], ascending=[False, True, True])
    .set_index(["Circuit type", "Method"])
    .style.format("{:.4f}")
)

draw_group_performance(
    circuit_type_plot_performance,
    "circuit_type_plot",
    circuit_type_order,
    f"Method Performance by Circuit Type ({EVAL_SPLIT.title()})",
    "performance_by_circuit_type",
)

## Regret and selection behavior

Reward regret is the configured per-circuit oracle reward minus the reward of the selected chain. Lower is better and zero means the method matched the oracle.

Selection counts are reported as the **average number of circuits per run**, so RL, Multiclass, and LLM remain comparable even when their run lists contain different numbers of models.

In [ ]:
regret_rows = []
selection_rows = []
selection_order = ["Original", "Single optimizer", "Two-step chain"]

for method, run_frames in method_run_frames.items():
    for run_index, frame in enumerate(run_frames):
        regret_rows.append(
            {
                "method": method,
                "run_index": run_index,
                "mean_regret": frame["regret_to_reward_oracle"].mean(),
                "median_regret": frame["regret_to_reward_oracle"].median(),
                "oracle_match_rate_pct": 100.0
                * np.isclose(frame["regret_to_reward_oracle"], 0.0, atol=1e-9).mean(),
            }
        )
        counts = frame["selection_type"].value_counts().reindex(selection_order, fill_value=0)
        for selection_type, count in counts.items():
            selection_rows.append(
                {
                    "method": method,
                    "run_index": run_index,
                    "selection_type": selection_type,
                    "count": int(count),
                    "selection_rate_pct": 100.0 * count / len(frame),
                }
            )

regret_per_run = pd.DataFrame(regret_rows)
regret_summary = (
    regret_per_run.groupby("method", as_index=False)
    .agg(
        runs=("run_index", "nunique"),
        mean_regret=("mean_regret", "mean"),
        run_to_run_regret_std=("mean_regret", lambda values: values.std(ddof=0)),
        median_regret=("median_regret", "mean"),
        oracle_match_rate_pct=("oracle_match_rate_pct", "mean"),
    )
)

selection_per_run = pd.DataFrame(selection_rows)
selection_summary = (
    selection_per_run.groupby(["method", "selection_type"], as_index=False)
    .agg(
        average_selections_per_run=("count", "mean"),
        run_to_run_count_std=("count", lambda values: values.std(ddof=0)),
        selection_rate_pct=("selection_rate_pct", "mean"),
    )
)

display(
    regret_summary.rename(
        columns={
            "method": "Method",
            "runs": "Runs",
            "mean_regret": "Mean reward regret",
            "run_to_run_regret_std": "Run-to-run regret std.",
            "median_regret": "Mean median regret",
            "oracle_match_rate_pct": "Oracle match rate (%)",
        }
    )
    .set_index("Method")
    .style.format("{:.4f}")
)
display(
    selection_summary.rename(
        columns={
            "method": "Method",
            "selection_type": "Selection type",
            "average_selections_per_run": "Average selections per run",
            "run_to_run_count_std": "Run-to-run count std.",
            "selection_rate_pct": "Selection rate (%)",
        }
    )
    .set_index(["Method", "Selection type"])
    .style.format("{:.3f}")
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
regret_plot = regret_summary.set_index("method").loc[["RL", "RF Classifier", "LLM"]].reset_index()
axes[0].bar(
    regret_plot["method"],
    regret_plot["mean_regret"],
    color=[METHOD_COLORS[name] for name in regret_plot["method"]],
)
axes[0].set_title("Mean Regret to Configured Reward Oracle")
axes[0].set_ylabel("Oracle reward - selected reward")
axes[0].set_xlabel("")
sns.despine(ax=axes[0])

selection_plot = (
    selection_summary.pivot(
        index="method",
        columns="selection_type",
        values="average_selections_per_run",
    )
    .reindex(index=["RL", "RF Classifier", "LLM"], columns=selection_order)
)
selection_plot.plot(
    kind="bar",
    stacked=True,
    color=["#A8A8A8", "#5B8E7D", "#C65D3B"],
    ax=axes[1],
)
axes[1].set_title("Average Chain-Type Selections per Run")
axes[1].set_ylabel("Evaluation circuits")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="")
sns.despine(ax=axes[1])

fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(FIGURE_DIR / "regret_and_selection_behavior.png", dpi=200, bbox_inches="tight")
    fig.savefig(FIGURE_DIR / "regret_and_selection_behavior.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Combined 2Q-reduction and RL-utility comparison
# Utility is the mean configured RL episode reward for the selected action/chain.
utility_rows = []
for method, run_frames in method_run_frames.items():
    utility_rows.append({
        "method": method,
        "utility": float(np.mean([frame["selected_reward"].mean() for frame in run_frames])),
    })

utility_rows.extend([
    {
        "method": "No Optimization",
        "utility": float(no_optimization["episode_reward_if_selected"].mean()),
    },
    {
        "method": "Best Fixed Reward Chain",
        "utility": float(best_fixed["episode_reward_if_selected"].mean()),
    },
    {
        "method": "Random Chain",
        "utility": float(
            eval_candidates.groupby("original_circuit")["episode_reward_if_selected"]
            .mean()
            .mean()
        ),
    },
    {
        "method": "Reward Oracle",
        "utility": float(reward_oracle["episode_reward_if_selected"].mean()),
    },
    {
        "method": "2Q Oracle",
        "utility": float(twoq_oracle["episode_reward_if_selected"].mean()),
    },
])
utility_table = pd.DataFrame(utility_rows)
utility_table["method"] = pd.Categorical(
    utility_table["method"], categories=METHOD_ORDER, ordered=True
)
utility_table = utility_table.sort_values("method").reset_index(drop=True)
utility_table["method"] = utility_table["method"].astype(str)

twoq_plot_table = pd.concat(
    [
        base_comparison[["method", "two_qubit_reduction_pct"]],
        pd.DataFrame([{"method": "2Q Oracle", "two_qubit_reduction_pct": metric_oracles["two_qubit_reduction_pct"]["two_qubit_reduction_pct"]}]),
    ],
    ignore_index=True,
)
twoq_plot_table["method"] = pd.Categorical(
    twoq_plot_table["method"], categories=METHOD_ORDER + ["2Q Oracle"], ordered=True
)
twoq_plot_table = twoq_plot_table.sort_values("method").reset_index(drop=True)
twoq_plot_table["method"] = twoq_plot_table["method"].astype(str)

combined_panel_table = twoq_plot_table.merge(utility_table, on="method", how="outer")
combined_panel_table["method"] = combined_panel_table["method"].replace({"Multiclass": "RF Classifier"})
display(
    combined_panel_table.rename(
        columns={
            "method": "Method",
            "two_qubit_reduction_pct": "2Q gate reduction (%)",
            "utility": "RL utility",
        }
    ).set_index("Method").style.format("{:.4f}")
)

panel_order = ["RL", "RF Classifier", "LLM", "Reward Oracle"]
panel_data = combined_panel_table.set_index("method").reindex(panel_order).reset_index()
for panel_index, (column, title, ylabel, filename, color) in enumerate([
    ("two_qubit_reduction_pct", f"2Q Gate Reduction on {EVAL_SPLIT.title()} Split", "Average 2Q gate reduction (%)", "two_qubit_reduction_only", "#4F7F8C"),
    ("utility", f"RL Utility on {EVAL_SPLIT.title()} Split", "Mean configured episode reward", "rl_utility_only", "#5B8E7D"),
]):
    fig, ax = plt.subplots(figsize=(8, 4.8), facecolor="white")
    plot_data = panel_data[["method", column]].sort_values(column, ascending=False).copy()
    bars = ax.barh(
        plot_data["method"],
        plot_data[column],
        color=color,
        edgecolor="#D0C8BC",
        linewidth=0.8,
        height=0.62,
        alpha=0.92,
    )
    ax.set_facecolor("#F7F4EE")
    ax.axvline(0.0, color="#444444", linewidth=0.8)
    ax.grid(axis="x", color="#D9D4CB", linewidth=0.7, alpha=0.75)
    ax.set_axisbelow(True)
    ax.set_title(title, fontsize=20, pad=10, weight="semibold")
    ax.text(
        -0.08, 1.06, f"({chr(97 + panel_index)})",
        transform=ax.transAxes,
        fontsize=13,
        fontweight="semibold",
        va="bottom",
        ha="right",
    )
    ax.set_xlabel(ylabel, fontsize=17, labelpad=7)
    ax.set_ylabel("")
    ax.tick_params(axis="x", labelsize=12)
    ax.tick_params(axis="y", labelsize=15, length=3, pad=6)
    ax.invert_yaxis()
    values = plot_data[column].to_numpy(dtype=float)
    value_range = float(np.nanmax(values) - np.nanmin(values))
    offset = 0.015 * (value_range if value_range > 0 else max(abs(values).max(), 1.0))
    for bar, value in zip(bars, values, strict=True):
        ax.text(
            value + offset if value >= 0 else value - offset,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.2f}",
            ha="left" if value >= 0 else "right",
            va="center",
            fontsize=11,
        )
    sns.despine(ax=ax)
    fig.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(FIGURE_DIR / f"{filename}.png", dpi=200, bbox_inches="tight")
        fig.savefig(FIGURE_DIR / f"{filename}.pdf", bbox_inches="tight")
    plt.show()